# 11. Current and combined configurations: validation and JECAM test

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import transfer, transfer_check, tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
test_path = ""  # Optional matching features_test.json; blank uses preprocessing 03
cohorts = list(transfer_check.COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
from crop_pipeline.training import notebook_workflow
test_file = notebook_workflow._features(root, test_path or None, test=True)
test = json.loads(Path(test_file).read_text())
print("Development features:", features_path)
print("Test features:", test_file)

In [ ]:
augmentation_run = ""  # Optional pipeline_outputs/transfer/augmentation_<id>; blank uses the latest
augmentation_run = Path(augmentation_run) if augmentation_run else tuning.latest_run(root, "augmentation", area=transfer.AREA)
choice = json.loads((augmentation_run / "selected.json").read_text())
print("augmentation run:", augmentation_run)
choice

## Current frozen models

In [ ]:
models_handoff, current = transfer_check.current_models(root)
print("Frozen models:", models_handoff)
from crop_pipeline.windows import seasons
provenance = json.loads((models_handoff.parent / "development.json").read_text())
for key in ("quality", "window_sha256"):
    if provenance[key] != test[key]:
        raise ValueError(f"Test features come from a different snapshot than the frozen models ({key})")
print("Frozen models trained on features with SHA-256", provenance["feature_sha256"][:12],
      "| this development file", seasons.file_sha256(features_path)[:12])

## Combined configuration

In [ ]:
settings = transfer_check.combined_settings(choice)
display(pd.DataFrame([{"cohort": c, **{k: str(v) for k, v in s.items()}} for c, s in settings.items()]))
import joblib
run = tuning.new_run(root, "final_check", features_path, area=transfer.AREA, augmentation_run=str(augmentation_run),
                     models_handoff=str(models_handoff), test_features=str(test_file))
combined = {}
for cohort in cohorts:
    model = transfer_check.fit_combined(development, cohort, settings[cohort])
    combined[cohort] = {"model": model, "variant": settings[cohort]["variant"]}
    (run / "models").mkdir(exist_ok=True)
    joblib.dump({"classifier": model, "variant": settings[cohort]["variant"],
                 "feature_columns": transfer.variant_columns(settings[cohort]["variant"]), "settings": settings[cohort]},
                run / "models" / f"{cohort}_combined.joblib")
    print(cohort, "fitted", flush=True)

## Locked validation

In [ ]:
records, parcel_sets = [], {}
for cohort in cohorts:
    train, validation = tuning.cohort_split(development, cohort)
    for name, entry in (("current", current[cohort]), ("combined", combined[cohort])):
        metrics, predicted = transfer_check.score_rows(entry["model"], validation, entry["variant"])
        parcel_sets["validation", cohort, name] = predicted
        records.append(transfer_check.metric_row("validation", cohort, name, metrics))

## JECAM external test

In [ ]:
manifests = [current[c]["manifest"] for c in cohorts]
for cohort in cohorts:
    train, _ = tuning.cohort_split(development, cohort)
    rows = transfer_check.test_rows(test, cohort, manifests)
    baseline = transfer_check.baseline_macro_f1(rows, transfer_check.majority_class(train))
    for name, entry in (("current", current[cohort]), ("combined", combined[cohort])):
        metrics, predicted = transfer_check.score_rows(entry["model"], rows, entry["variant"])
        parcel_sets["jecam_test", cohort, name] = predicted
        records.append(transfer_check.metric_row("jecam_test", cohort, name, metrics, baseline))
        seasons.write_records(run / f"jecam_{cohort}_{name}_parcels.csv", predicted)
tuning.save_records(run, "scores", records)
display(pd.DataFrame(records).round(3))

## Check against the Testing 01 report

In [ ]:
from crop_pipeline.analysis_workflow import resolve_stage_pointer
scores_handoff = resolve_stage_pointer(root / "pipeline_outputs/testing/current_scores.json")
reported = json.loads((scores_handoff.parent / "reports/test_scores.json").read_text())
check = [{"cohort": c, "testing_01_macro_f1": reported[c].get("macro_f1"),
          "recomputed_current_macro_f1": next(r["macro_f1"] for r in records
                                              if r["stage"] == "jecam_test" and r["cohort"] == c and r["configuration"] == "current")}
         for c in cohorts]
display(pd.DataFrame(check).round(4))

## Paired bootstrap: combined minus current

In [ ]:
differences = []
for stage in ("validation", "jecam_test"):
    for cohort in cohorts:
        result = transfer_check.paired_bootstrap(parcel_sets[stage, cohort, "current"], parcel_sets[stage, cohort, "combined"])
        differences.append({"stage": stage, "cohort": cohort, **result})
tuning.save_records(run, "paired_bootstrap", differences)
display(pd.DataFrame(differences).round(3))

## Confusion matrices

In [ ]:
for (stage, cohort, name), predicted in parcel_sets.items():
    if stage != "jecam_test":
        continue
    table = pd.crosstab(pd.Series([r["crop"] for r in predicted], name="actual"),
                        pd.Series([r["prediction"] for r in predicted], name="predicted"))
    print(stage, cohort, name)
    display(table)

## Figure

In [ ]:
table = pd.DataFrame(records)
figure, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for axis, stage in zip(axes, ["validation", "jecam_test"]):
    table[table.stage == stage].pivot(index="cohort", columns="configuration", values="macro_f1").plot(kind="bar", ax=axis)
    if stage == "jecam_test":
        baseline = table[(table.stage == stage) & (table.configuration == "current")].set_index("cohort").majority_baseline_macro_f1
        axis.scatter(range(len(baseline)), baseline.values, color="black", marker="_", s=400, label="majority baseline", zorder=3)
        axis.legend()
    axis.set_title(stage)
    axis.set_ylim(0, 1)
    axis.set_ylabel("Parcel macro-F1")
show_and_save(figure, run, "validation_and_test")
print("Run folder:", run)